# Man-in-the-Middle — Stealth Hijack

This is the same attacker-driven redirect as `11-mitm_hijack.ipynb` (an
`InterventionStrategy` run from the man-in-the-middle position), but with the
MITM's **stealth** flag enabled (`stealth=True`).

In the visible hijack, the instant the attacker's intervention switches the
vehicle to GUIDED the `HEARTBEAT.custom_mode` flips from **3 (AUTO)** to
**4 (GUIDED)**, and because telemetry still flows to the GCS, that switch is
recorded in the GCS's telemetry log — a clear fingerprint of the compromise.

With `stealth=True`, the MITM applies a downlink **mode mask** (see
`StealthModeMask`): once the runner takes control it freezes the last mode the
vehicle reported before the hijack fired and rewrites every subsequent downlink
`HEARTBEAT` to keep reporting that cover mode. It also **drops any downlink
`STATUSTEXT` that names the hijack mode** (`GUIDED`) — a second tell some
firmwares (ArduPlane, some copter configs) emit on a mode change. The GCS still
receives a live heartbeat stream (so the link never looks dead), but it never
sees the hijack's switch to GUIDED. This is the downlink analogue of the uplink
`GCS_FORWARD_TYPES` allowlist — instead of choosing which message *types* pass,
it scrubs the fields (and drops the texts) that would betray the attack.

Note that GUIDED is **not** inherently a red flag: the `AutoPlan` takeoff runs
in GUIDED before the mission switches to AUTO, so the GCS legitimately sees
GUIDED early on. The mask only engages when the hijack fires, so the last cell
checks for a leak *within the hijack window* (after the mission reached the
trigger waypoint) rather than naively scanning the whole log.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import Intervention, MissionTrigger, SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.helpers.processes import SimProcess
from simulator.planner import AutoPlan, InterventionPlan
from simulator.runtime.mitm import InterventionStrategy
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)
speed = 5.0        # m/s
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

# Mission seq: seq=0 home, seq=1 TAKEOFF, seq=2 flying to north_100,
#              seq=3 flying to north_200 ← attacker hijacks here
home_wp    = ENU(x=0, y=0,   z=0)
north_100  = ENU(x=0, y=100, z=cruise_alt)
north_200  = ENU(x=0, y=200, z=cruise_alt)
mission_wps = [home_wp, north_100, north_200]
trigger_seq = 3

# Attacker's destination, as an ENU relative to the run origin (100 m WEST,
# opposite the mission). The InterventionPlan converts it to geodetic when it
# binds inside the MITM.
hijack_target = ENU(x=-100, y=0, z=cruise_alt)
print(f"Hijack target (ENU): {hijack_target}")

## Vehicle + MITM stealth hijack

In [ ]:
mission_folder = DATA_PATH / "missions"
mission_folder.mkdir(parents=True, exist_ok=True)

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=str(mission_folder / "mitm_stealth.waypoints"),
    navigation_speed=speed,
    firmware=model.firmware,
)

# Same hijack as 11-mitm_hijack.ipynb, but `stealth=True` masks the GUIDED-mode
# switch from the GCS. Set `stealth=False` to reproduce the visible hijack and
# see custom_mode=4 appear in the GCS log for comparison.
hijack = InterventionStrategy(
    Intervention(
        trigger=MissionTrigger(seq=trigger_seq),
        plan=InterventionPlan.from_relative_path(
            relative_path=[hijack_target],
            enu_origin=enu_origin,
            firmware=model.firmware,
            land=False,
        ),
        firmware=model.firmware,
    ),
    stealth=True,
)

vehicle = SimVehicle.from_relative(
    model=model,
    sysid=sysid,
    color=Color.BLUE,
    plan=plan,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    mitm=hijack,
)

gcs = SimGCS(name=f"BLUE_{Color.BLUE.emoji}")
gcs.own(vehicle)

## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
target_marker = GazMarker(
    name="hijack_target",
    group="targets",
    pos=hijack_target,
    color=Color.RED,
)
gaz.markers.append(target_marker)
gaz.markers.append(origin_marker)

## Oracle

In [ ]:
orac = Oracle()
orac.add_vehicle(vehicle)

## Simulator + MITM stealth hijack

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    verbose=2,
    terminals=[SimProcess.MITM, SimProcess.LOGIC, SimProcess.GCS],
    speedup=1,
)
simulator.preview()

In [ ]:
simulator.run(timeout=120)

## Did the GCS ever see the hijack?

The GCS logs every message it receives to `simulator/data/gcs_msgs/veh_<sysid>.jsonl`.
We pull `custom_mode` out of every recorded `HEARTBEAT` and split them into two
phases, because **GUIDED appears legitimately before the attack**: ArduCopter's
`AutoPlan` arms and takes off in GUIDED (4), then switches to AUTO (3) for the
waypoint run. A real GCS operator expects that takeoff GUIDED, so counting it as
a leak would be a false positive.

The hijack's fingerprint is leaving AUTO *after the mission is underway*. We
therefore scope the leak check to the **hijack window** — heartbeats recorded
once the GCS itself observed `MISSION_CURRENT.seq >= trigger_seq` (the same
condition that fires the attack). With stealth on, every heartbeat in that
window should still report AUTO (3) — never GUIDED (4) — even though the vehicle
really did switch to GUIDED and fly west.

In [ ]:
import json

AUTO, GUIDED = 3, 4
GUIDED_NAME = "GUIDED"
log_path = DATA_PATH / "gcs_msgs" / f"veh_{sysid}.jsonl"

# The vehicle legitimately passes through GUIDED during the AutoPlan *takeoff*
# (ArduCopter arms/takes off in GUIDED, then switches to AUTO for the waypoint
# run), long before the attacker fires. That benign GUIDED is expected, so a
# naive "did GUIDED ever appear" check gives a false positive. The hijack's
# signature is leaving AUTO *after* the mission is underway — i.e. once the GCS
# has itself seen MISSION_CURRENT.seq >= trigger_seq, the same condition that
# fires the hijack. We therefore scope the leak check to that hijack window.
#
# Two downlink tells are checked:
#   1. HEARTBEAT.custom_mode flipping to GUIDED (4);
#   2. a mode-change STATUSTEXT naming GUIDED (some firmwares emit one).
# Stealth rewrites (1) back to the cover mode and drops (2).
pre_modes: list[int] = []
hijack_modes: list[int] = []
hijack_mode_texts: list[str] = []
in_hijack_window = False
with log_path.open() as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        msg_type = record.get("msg_type")
        if msg_type == "MISSION_CURRENT":
            if int(record["data"]["seq"]) >= trigger_seq:
                in_hijack_window = True
        elif msg_type == "HEARTBEAT":
            mode = int(record["data"]["custom_mode"])
            (hijack_modes if in_hijack_window else pre_modes).append(mode)
        elif msg_type == "STATUSTEXT" and in_hijack_window:
            text = str(record["data"].get("text", ""))
            if GUIDED_NAME in text.upper():
                hijack_mode_texts.append(text)

pre_seen = sorted(set(pre_modes))
hijack_seen = sorted(set(hijack_modes))
hb_leak = GUIDED in hijack_seen
text_leak = bool(hijack_mode_texts)

print(f"Pre-hijack heartbeats (takeoff + AUTO cruise): {len(pre_modes)}")
print(f"  custom_mode values seen: {pre_seen}"
      f"{'  (includes benign takeoff GUIDED)' if GUIDED in pre_seen else ''}")
print(f"Hijack-window heartbeats (after GCS saw seq >= {trigger_seq}): {len(hijack_modes)}")
print(f"  custom_mode values seen: {hijack_seen}")
print(f"  HEARTBEAT leaked GUIDED ({GUIDED})?        {hb_leak}")
print(f"  STATUSTEXT leaked a GUIDED mode message?  {text_leak}"
      f"{'  -> ' + str(hijack_mode_texts) if text_leak else ''}")

if hb_leak or text_leak:
    print("\n⚠️  The hijack leaked: GUIDED reached the GCS (stealth off?).")
else:
    print("\n✅ Stealth held: once the hijack fired, the GCS only ever saw AUTO —")
    print("   no GUIDED heartbeat and no GUIDED mode-change text got through.")

## What to observe

- **In Gazebo / on the real trajectory** the drone still flies north then turns
  **west** mid-mission — the hijack physically happened.
- **In the GCS telemetry log** the heartbeats keep reporting AUTO (3) throughout
  the hijack window, and no GUIDED mode-change `STATUSTEXT` gets through. The
  analysis cell above confirms GUIDED (4) never reached the GCS *after the attack
  fired* (the only GUIDED heartbeats are the benign takeoff ones, before the
  mission reached the trigger waypoint).
- `simulator/logs/mitm/mitm_1.log` — `MITM intervention: taking over vehicle 1`
  (the shared `InterventionRunner` running inside the MITM), plus
  `MITM stealth: masking flight mode as base=... custom=3` and (on firmwares
  that announce the mode) `MITM stealth: dropping mode-change STATUSTEXT ...`.
- Flip `stealth` to `False` in the MITM cell and re-run to watch GUIDED (4)
  reappear in the hijack window of the GCS log — the visible hijack of
  `11-mitm_hijack.ipynb`.